# Preparación del entorno local

Dependencias: `python -m pip install pandas pyarrow jupyter ipykernel`.

Abre Jupyter desde la raíz del repositorio, que debe contener `data/planuno`, `data/customers_ba` y `data/bajas2021`. También puedes ajustar `PROJECT_ROOT` en la siguiente celda.

Se conservan las etapas y la fecha de referencia originales. Las salidas antiguas se han eliminado para evitar confundirlas con resultados de esta versión. Usa una carpeta de salida nueva si tienes exportaciones antiguas de Spark.


In [1]:
# Importar librerías necesarias
from pathlib import Path
import pandas as pd


In [ ]:
# Ejecuta Jupyter desde la raíz del repositorio.
# Si el notebook está en otra carpeta, indica aquí la raíz del proyecto.
PROJECT_ROOT = Path.cwd()
base_path = PROJECT_ROOT / "data"


In [ ]:
# Left join con el comportamiento de Spark para claves nulas.
def left_join(izquierda, derecha):
    derecha = derecha.loc[derecha["customer_id"].notna()]
    return izquierda.merge(derecha, on="customer_id", how="left", sort=False)


In [ ]:
ruta_output = PROJECT_ROOT / "output" / "caso_de_uso_3"


# Lectura de todos los .parquet

In [ ]:
# Rutas locales: archivos o carpetas Parquet.
ruta1 = base_path / "planuno"
ruta2 = base_path / "customers_ba"
ruta3 = base_path / "bajas2021"

df_plauno = pd.read_parquet(ruta1)
df_customers = pd.read_parquet(ruta2)
df_bajas = pd.read_parquet(ruta3)

fechas = pd.to_datetime(df_plauno["date_alta"], errors="coerce")
planuno_2020 = df_plauno.loc[fechas < pd.Timestamp("2021-01-01")].copy()
planuno_2021 = df_plauno.loc[fechas >= pd.Timestamp("2021-01-01")].copy()


In [ ]:
df_plauno.head(20)

In [ ]:
df_plauno[["planuno_quadrant_name"]].drop_duplicates()

# Base de clientes Basicos

In [ ]:
condicion = (
    df_plauno["business_area_id"].eq("BA03")
    & df_plauno["planuno_quadrant_name"].eq("Básico")
)
base = (
    df_plauno.loc[condicion.fillna(False), ["customer_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
base.head(20)


# Antiguedad del cliente

In [ ]:
clientes = df_customers[["customer_id", "cust_entry_date"]].rename(
    columns={"cust_entry_date": "fecha_alta_cliente"}
)
df_antiguedad = left_join(base, clientes)
# Conservamos la fecha del notebook original; el engine usa 2022-06-25.
df_antiguedad["hoy"] = "2022-01-30"
fechas_alta = pd.to_datetime(
    df_antiguedad["fecha_alta_cliente"], errors="coerce"
).dt.normalize()
df_antiguedad["diferencia_dias"] = (
    pd.Timestamp("2022-01-30") - fechas_alta
).dt.days.astype("Int64")

dias = df_antiguedad["diferencia_dias"]
df_antiguedad["antiguedad"] = "> 5 años"
for limite, etiqueta in [
    (1825, "2-5 años"), (730, "1-2 años"),
    (365, "6-12 meses"), (183, "< 6 meses"),
]:
    df_antiguedad.loc[dias.lt(limite).fillna(False), "antiguedad"] = etiqueta

df_antiguedad.head(20)


# Refunidos

In [ ]:
refunidos_df = planuno_2020[["customer_id", "date_alta"]].copy()
refunidos_df["_fecha_orden"] = pd.to_datetime(
    refunidos_df["date_alta"], errors="coerce"
)
refunidos_df = (
    refunidos_df.sort_values(
        "_fecha_orden", ascending=False, na_position="last", kind="stable"
    )
    .drop_duplicates(subset="customer_id", keep="first")
    .drop(columns="_fecha_orden")
    .reset_index(drop=True)
)
refunidos_df.head(20)


# Clasificacion Plan Uno 2021 + flag 3d3

In [ ]:
# Conservamos el uso de planuno_2020 y los nombres originales.
particulares = (
    planuno_2020["customer_type"].isin(
        ["B.PRIVADA", "B.PERSONAL", "PARTICULARES", "PAES"]
    )
    & planuno_2020["mobile_digital_crit_type"].eq(1)
    & planuno_2020["income_freelance_type"].eq(0)
    & planuno_2020["transac_business_type"].eq(0)
)
suma_criterios = (
    planuno_2020["soc_insur_asgn_fulflt_type"]
    + planuno_2020["payroll_asgn_crit_type"]
    + planuno_2020["tax_pymt_crit_type"]
)
pymes = (
    planuno_2020["customer_type"].eq("PYMES")
    & planuno_2020["mobile_digital_crit_type"].eq(1)
    & suma_criterios.eq(1)
    & planuno_2020["planuno_segment_id"].isin(["PT", "AT", "CT", "BT", "VT"])
)
planuno_2021_enriq = planuno_2020[
    ["customer_id", "planuno_quadrant_name"]
].copy()
planuno_2021_enriq["Exclusivos_3d3"] = "Otros"
planuno_2021_enriq.loc[
    (particulares | pymes).fillna(False), "Exclusivos_3d3"
] = "Exclusivos_3d3"
planuno_2021_enriq.head(20)


In [ ]:
planuno_2020.head(2)

# Union con clasificacion 2021 y bajas

In [ ]:
df = left_join(df_antiguedad, planuno_2021_enriq)
df = left_join(df, df_bajas[["customer_id", "movement_type"]])
df["bajas"] = df["movement_type"].eq("baja").fillna(False).astype(int)
df.head(20)


# Columna cuadrante_planuno

In [ ]:
df_cuadrante_planuno = df.copy()
df_cuadrante_planuno["cuadrante_planuno"] = df_cuadrante_planuno[
    "planuno_quadrant_name"
].copy()
condicion_3d3 = (
    df_cuadrante_planuno["planuno_quadrant_name"].eq("Transaccional")
    & df_cuadrante_planuno["Exclusivos_3d3"].eq("Exclusivos_3d3")
)
df_cuadrante_planuno.loc[
    condicion_3d3.fillna(False), "cuadrante_planuno"
] = "3d3"
df_cuadrante_planuno.loc[
    df_cuadrante_planuno["bajas"].eq(1).fillna(False), "cuadrante_planuno"
] = "BAJAS"
df_cuadrante_planuno.head(20)


# Resultado

In [ ]:
df_cuadrante_planuno[
    ["customer_id", "antiguedad", "cuadrante_planuno", "Exclusivos_3d3", "bajas"]
].head(20)


In [ ]:
(
    df_cuadrante_planuno
    .groupby(["antiguedad", "cuadrante_planuno"], dropna=False, observed=True)
    .size()
    .reset_index(name="count")
    .sort_values(["antiguedad", "cuadrante_planuno"], na_position="first")
)


# Exportar csv a output

In [ ]:
# Exportación a un archivo dentro de una carpeta, sin cabecera,
# como la exportación CSV original de Spark.
csv_path_pyspark = ruta_output / "cuadrante_planuno"
csv_path_panda = ruta_output / "cuadrante_planuno.csv"
csv_path_pyspark.mkdir(parents=True, exist_ok=True)
df_cuadrante_planuno.to_csv(
    csv_path_pyspark / "resultado.csv", index=False, header=False
)


In [ ]:
# Exportación a un único CSV con cabecera.
ruta_output.mkdir(parents=True, exist_ok=True)
df_cuadrante_planuno.to_csv(csv_path_panda, index=False)
print(f"Resultado guardado en: {csv_path_panda}")
